In [ ]:
# Epoch-Based Extreme Rainfall Frequency Comparison

**Purpose:**  
Quantify changes in extreme rainfall event frequency across historical time windows (“epochs”) using observed PRISM data.

This notebook implements the same epoch-based comparison logic used in the thesis analysis, generalized for replication and reuse across regions.

**Key Concept:**  
Climate non-stationarity is expressed here as **changes in event frequency over time**, not shifts in intensity or fitted return-period models.

**Inputs:**  
- Annual extreme-event counts generated in `02_prism_extremes_analysis.ipynb`

**Outputs:**  
- Epoch-level event totals  
- Absolute and percent frequency change metrics  
- Tables suitable for mapping, reporting, and comparison


In [ ]:
## Interpretation Notes

- Epoch comparisons reflect **observed changes in event frequency**, not probabilistic return periods.
- Percent change values are sensitive to low baseline counts and should be interpreted alongside absolute totals.
- This step preserves the empirical framing of the original thesis while improving transparency and reuse.

Results from this notebook feed into:
- flood probability context (Notebook 05)
- exposure and vulnerability overlays (Notebook 04)


In [ ]:
from pathlib import Path
import pandas as pd
REPO_ROOT = Path("..").resolve()
OUTPUTS_DIR = REPO_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"

INPUT_PATH = TABLES_DIR / "prism_annual_extreme_counts.csv"
OUTPUT_PATH = TABLES_DIR / "epoch_comparison_summary.csv"


In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing annual extremes table at {INPUT_PATH}. "
        "Run 02_prism_extremes_analysis.ipynb first."
    )

df = pd.read_csv(INPUT_PATH, parse_dates=["year"])
df.head()

In [ ]:
# Epoch configuration
EPOCH_YEARS = 20

START_YEAR = df["year"].dt.year.min()
END_YEAR = df["year"].dt.year.max()


In [ ]:
df["epoch_start"] = (
    ((df["year"].dt.year - START_YEAR) // EPOCH_YEARS) * EPOCH_YEARS
) + START_YEAR

epoch_summary = (
    df.groupby(["region_id", "epoch_start"], as_index=False)
      .agg(epoch_event_total=("extreme_event_count", "sum"))
)


In [ ]:
epoch_summary = epoch_summary.sort_values(
    ["region_id", "epoch_start"]
)

epoch_summary["prev_epoch_total"] = (
    epoch_summary.groupby("region_id")["epoch_event_total"].shift(1)
)

epoch_summary["percent_change"] = (
    (epoch_summary["epoch_event_total"] - epoch_summary["prev_epoch_total"])
    / epoch_summary["prev_epoch_total"]
) * 100


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
epoch_summary.to_csv(OUTPUT_PATH, index=False)

print(f"Epoch comparison table saved to {OUTPUT_PATH}")


In [ ]:
## Next Steps

This notebook compares annual extreme rainfall event counts across historical time windows (“epochs”) to identify observed changes in event frequency without assuming stationarity.

The epoch-based frequency change metrics generated here are used in:

- `04_cordex_future_frequency.ipynb`  
  → to compare observed frequency shifts with projected future changes from climate models

Users may proceed directly to the next notebook after confirming this step completes successfully.
